<a href="https://colab.research.google.com/github/f-ai0/ds-training/blob/main/week7/Day1_Tensors_Autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1.1 — Tensors and Device Handling

In [1]:
!pip install -q torch torchvision

In [2]:
import torch
import numpy as np

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch version: 2.11.0+cpu
CUDA available: False
Using device: cpu


In [3]:
# إنشاء tensors بطرق مختلفة
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros(2, 3)
c = torch.randn(2, 3)          # random normal
d = torch.arange(0, 10, 2)

print(f'a: {a}, shape: {a.shape}, dtype: {a.dtype}')
print(f'c:\n{c}')

a: tensor([1., 2., 3.]), shape: torch.Size([3]), dtype: torch.float32
c:
tensor([[-0.6748, -1.6265,  0.2730],
        [-0.4882, -0.9714,  0.9034]])


In [4]:
# التحويل بين NumPy وPyTorch
np_array = np.array([[1, 2], [3, 4]])
tensor_from_np = torch.from_numpy(np_array)
back_to_np = tensor_from_np.numpy()

print(f'From NumPy: {tensor_from_np}')
print(f'Back to NumPy: {back_to_np}')

From NumPy: tensor([[1, 2],
        [3, 4]])
Back to NumPy: [[1 2]
 [3 4]]


In [5]:
# نقل tensor إلى GPU (إذا متوفر)
c_gpu = c.to(device)
print(f'Tensor device: {c_gpu.device}')

Tensor device: cpu


# Task 1.2 — Autograd: Automatic Differentiation

In [6]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1     # y = x^2 + 2x + 1

y.backward()               # حساب dy/dx

print(f'x = {x.item()}')
print(f'y = {y.item()}')
print(f'dy/dx = {x.grad.item()}')
# بالحساب اليدوي: dy/dx = 2x + 2 = 2(3) + 2 = 8  -> مطابق!

x = 3.0
y = 16.0
dy/dx = 8.0


In [7]:
# مع vectors
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
loss = (w ** 2).sum()
loss.backward()
print(f'Gradients: {w.grad}')   # d/dw لـ w^2 = 2w = [2, 4, 6]

Gradients: tensor([2., 4., 6.])


In [8]:
# مهم: الـ gradients تتراكم. لازم تصفرينها بين كل خطوة وأخرى.
w.grad.zero_()
print(f'After zero_(): {w.grad}')

After zero_(): tensor([0., 0., 0.])


# Task 1.3 — Manual Gradient Descent

In [9]:
# بيانات وهمية: العلاقة الحقيقية هي y = 3x + 2
X = torch.randn(100, 1)
y_true = 3 * X + 2 + 0.1 * torch.randn(100, 1)

w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

learning_rate = 0.1

for epoch in range(100):
    # 1. Forward pass
    y_pred = w * X + b

    # 2. Compute loss
    loss = ((y_pred - y_true) ** 2).mean()

    # 3. Backward pass
    loss.backward()

    # 4. Update parameters
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    # 5. Zero the gradients
    w.grad.zero_()
    b.grad.zero_()

    if epoch % 20 == 0:
        print(f'Epoch {epoch}: loss={loss.item():.4f}, w={w.item():.3f}, b={b.item():.3f}')

print(f'\nLearned: y = {w.item():.2f}x + {b.item():.2f}  (true: y = 3x + 2)')

Epoch 0: loss=13.2400, w=0.281, b=0.562
Epoch 20: loss=0.0133, w=2.967, b=1.964
Epoch 40: loss=0.0075, w=3.011, b=2.013
Epoch 60: loss=0.0075, w=3.012, b=2.014
Epoch 80: loss=0.0075, w=3.012, b=2.014

Learned: y = 3.01x + 2.01  (true: y = 3x + 2)
